In [3]:
import pandas as pd
from sqlalchemy import create_engine, text

In [ ]:
server = '.'
database = 'GrocerySales_Staging_Database'
engine = create_engine(f'mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server')
categories = pd.read_csv('F:\Self Study Projects\SQL\Grocery Sales\categories.csv')
products = pd.read_csv('F:\Self Study Projects\SQL\Grocery Sales\products.csv')
cities = pd.read_csv('F:\Self Study Projects\SQL\Grocery Sales\cities.csv')
countries = pd.read_csv('F:\Self Study Projects\SQL\Grocery Sales\countries.csv')
employees = pd.read_csv('F:\Self Study Projects\SQL\Grocery Sales\employees.csv')
customers = pd.read_csv('F:\Self Study Projects\SQL\Grocery Sales\customers.csv')

tabs_dict = {
    'categories' : categories,
    'products' : products,
    'cities' : cities,
    'countries' : countries,
    'employees' : employees,
    'customers' : customers
}
for i, j in tabs_dict.items():
    j.to_sql(i, con = engine, if_exists = 'append', index = False)
    print(f'{i} is imported successfully')
print('all tables are imported successfully')


In [ ]:
sales = pd.read_csv('F:\Self Study Projects\SQL\Grocery Sales\sales.csv')
sales.to_sql('sales', con = engine, if_exists = 'append', index = False)

In [4]:
def connections():
        server = '.'
        database = 'GrocerySalesStage'
        dwh = 'GrocerySales_DWH'
        source_engine = create_engine(f'mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server')
        dwh_engine = create_engine(f'mssql+pyodbc://{server}/{dwh}?driver=ODBC+Driver+17+for+SQL+Server')
        return source_engine, dwh_engine

In [5]:
def extract_dim(source_engine):
        print("=== Starting Dimensions Extraction ===")
        
        customers_df  = pd.read_sql('select * from GrocerySalesStage.dbo.customers', source_engine)
        employees_df  = pd.read_sql('select * from GrocerySalesStage.dbo.employees', source_engine)
        countries_df  = pd.read_sql('select * from GrocerySalesStage.dbo.countries', source_engine)
        cities_df     = pd.read_sql('select * from GrocerySalesStage.dbo.cities', source_engine)
        products_df   = pd.read_sql('select * from GrocerySalesStage.dbo.products', source_engine)
        categories_df = pd.read_sql('select * from GrocerySalesStage.dbo.categories', source_engine)

        return customers_df, employees_df, countries_df, cities_df, products_df, categories_df


In [6]:
def transform_dim(customers_df, employees_df, countries_df, cities_df, products_df, categories_df):
        print("\n=== Starting Dimensions Transformation ===")
        
        cities_df['CountryID'] = cities_df['CountryID'].astype(int)
        countries_df['CountryID'] = countries_df['CountryID'].astype(int)
        categories_df['CategoryID'] = categories_df['CategoryID'].astype(int)
        products_df['CategoryID'] = products_df['CategoryID'].astype(int)

        dim_customers = customers_df.copy()
        dim_employees = employees_df.copy()
        dim_cities    = cities_df.merge(countries_df, on='CountryID', how='left')
        dim_products  = products_df.merge(categories_df, on='CategoryID', how='left')
        
        dim_customers = dim_customers.drop('CityID', axis=1)
        dim_employees = dim_employees.drop('CityID', axis=1)
        dim_employees['BirthDate'] = pd.to_datetime(dim_employees['BirthDate']).dt.date
        dim_employees['HireDate']  = pd.to_datetime(dim_employees['HireDate']).dt.date
        dim_cities    = dim_cities.drop('CountryCode', axis=1)
        dim_products['VitalityDays'] = dim_products['VitalityDays'].astype(float).astype(int)

        return dim_customers, dim_employees, dim_cities, dim_products
    

In [7]:
def scd_type_1(source_table, dwh_dim, dim_bk, column_mapping, dwh_engine):
    print(f"\n--- Loading {dwh_dim} (SCD Type 1) ---")

    df_renamed = source_table.rename(columns=column_mapping)
    staging_temp_table = f"#temp_{dwh_dim}"
    df_renamed.to_sql(staging_temp_table, dwh_engine, if_exists='replace', index=False)
            
    update_set     = ', '.join([f"target.{col} = source.{col}" for col in df_renamed.columns if col != dim_bk])
    insert_columns = ', '.join(df_renamed.columns)
    insert_values  = ', '.join([f"source.{col}" for col in df_renamed.columns])
            
    merge_query = f"""
        MERGE {dwh_dim} AS target
        USING {staging_temp_table} AS source
        ON target.{dim_bk} = source.{dim_bk}
        WHEN MATCHED THEN
            UPDATE SET {update_set}
        WHEN NOT MATCHED THEN
            INSERT ({insert_columns})
            VALUES ({insert_values});
    """
            
    with dwh_engine.connect() as conn:
        conn.execute(text(merge_query))
        conn.commit()

In [8]:
def scd_type_2(source_table, dwh_dim, source_bk, column_mapping, dwh_engine):
            print(f"\n--- Loading {dwh_dim} (SCD Type 2) ---")

            df_renamed = source_table.rename(columns=column_mapping)
            staging_temp_table = f"#temp_{dwh_dim}"
            df_renamed.to_sql(staging_temp_table, dwh_engine, if_exists='replace', index=False)
            
            mapped_bk     = column_mapping[source_bk]
            compare_columns = [col for col in df_renamed.columns if col != mapped_bk]
            match_conditions = ' OR '.join([
                f"(target.{col} != source.{col} OR "
                f"(target.{col} IS NULL AND source.{col} IS NOT NULL) OR "
                f"(target.{col} IS NOT NULL AND source.{col} IS NULL))"
                for col in compare_columns
            ])
            
            insert_columns = ', '.join(df_renamed.columns)
            insert_values  = ', '.join([f"source.{col}" for col in df_renamed.columns])
            
            merge_query = f"""
                UPDATE target
                SET is_current = 0, end_time = GETDATE()
                FROM {dwh_dim} target
                INNER JOIN {staging_temp_table} source ON target.{mapped_bk} = source.{mapped_bk}
                WHERE target.is_current = 1 AND ({match_conditions});
                
                MERGE {dwh_dim} AS target
                USING {staging_temp_table} AS source
                ON target.{mapped_bk} = source.{mapped_bk} AND target.is_current = 1
                WHEN NOT MATCHED THEN
                    INSERT ({insert_columns})
                    VALUES ({insert_values});
            """
            
            with dwh_engine.connect() as conn:
                conn.execute(text(merge_query))
                conn.commit()

In [9]:
def load_dim(dim_customers, dim_employees, dim_cities, dim_products, dwh_engine):
    print("\n=== Starting Dimensions Load ===")
    
    customers_mapping = {
        'CustomerID':'customer_pk',
        'FirstName':'first_name',
        'MiddleInitial':'middle_initial',
        'LastName':'last_name',
        'Address':'address'
    }
    employees_mapping = {
        'EmployeeID':'employee_pk',
        'FirstName':'first_name',
        'MiddleInitial':'middle_initial',
        'LastName':'last_name',
        'BirthDate':'birth_date',
        'Gender':'gender',
        'HireDate':'hire_date'
    }
    cities_mapping = {
        'CityID':'city_id',
        'CityName':'city_name',
        'Zipcode':'zip_code',
        'CountryID':'country_pk',
        'CountryName':'country_name'
    }
    products_mapping = {
        'ProductID':'product_pk',
        'ProductName':'product_name',
        'Price':'price',
        'CategoryID':'category_id',
        'Class':'class',
        'ModifyDate':'modify_date',
        'Resistant':'resistant',
        'IsAllergic':'is_allergic',
        'VitalityDays':'vitality_days',
        'CategoryName':'category_name'
    }
    scd_type_1(dim_customers, 'Dim_Customers', 'customer_pk', customers_mapping, dwh_engine)
    scd_type_1(dim_employees, 'Dim_Employees', 'employee_pk', employees_mapping, dwh_engine)
    scd_type_1(dim_cities,    'Dim_Cities', 'city_id', cities_mapping, dwh_engine)
    scd_type_2(dim_products,  'Dim_Products',  'ProductID', products_mapping, dwh_engine)
        
    print("\n=== Dimensions are Loaded Successfully ===")

In [10]:
def get_last_loaded_sales_id(source_engine):
    """get max sales_pk that is already loaded into Fact_Sales"""

    query = """
                select isnull(max(sales_pk),0) from GrocerySales_DWH.dbo.Fact_Sales                
            """
    result = pd.read_sql(query, source_engine)
    return result.iloc[0, 0]

In [11]:
def get_total_rows_to_process(last_id, source_engine):
    """Get count of rows to process"""
    
    count_query = f"""
        select count(*) 
        from GrocerySalesStage.dbo.sales s
        where s.SalesID > {last_id}
    """
    result = pd.read_sql(count_query, source_engine)
    return result.iloc[0, 0]

In [12]:
def load_dimension_lookups(dwh_engine):
    """Load dimension lookup tables once and reuse for all batches"""
        
    print("Loading dimension lookup tables...")
    dim_customers_loaded = pd.read_sql('SELECT customer_sk, customer_pk FROM Dim_Customers', dwh_engine)
    dim_employees_loaded = pd.read_sql('SELECT employee_sk, employee_pk FROM Dim_Employees', dwh_engine)
    dim_cities_loaded    = pd.read_sql('SELECT country_city_sk, city_id FROM Dim_Cities', dwh_engine)
    dim_products_loaded  = pd.read_sql('SELECT product_sk, product_pk FROM Dim_Products WHERE is_current = 1', dwh_engine)
        
    return {
        'customers': dim_customers_loaded,
        'employees': dim_employees_loaded,
        'cities': dim_cities_loaded,
        'products': dim_products_loaded
    }

In [13]:
def transform_fact_chunk(chunk, dim_lookups):
    """Transform a single chunk of sales data"""
    # Convert data types
    chunk['SalesID'] = chunk['SalesID'].astype(float).astype(int)
    chunk['SalesPersonID'] = chunk['SalesPersonID'].astype(float).astype(int)
    chunk['CustomerID'] = chunk['CustomerID'].astype(float).astype(int)
    chunk['CityID'] = chunk['CityID'].astype(float).astype(int)
    chunk['ProductID'] = chunk['ProductID'].astype(float).astype(int)
    chunk['Quantity'] = chunk['Quantity'].astype(float)
    chunk['Price'] = chunk['Price'].astype(float)
    chunk['Discount'] = chunk['Discount'].astype(float)
    chunk['total_sales'] = chunk['Price'] * chunk['Quantity'] * (1 - chunk['Discount'])

    #   Merge with dimension  loo   kups
    chunk = chunk.merge(dim_lookups['customers'], left_on='CustomerID', right_on='customer_pk', how='left')\
    .merge(dim_lookups['employees'], left_on='SalesPersonID', right_on='employee_pk', how='left')\
    .merge(dim_lookups['cities'], left_on='CityID', right_on='city_id', how='left')\
    .merge(dim_lookups['products'], left_on='ProductID', right_on='product_pk', how ='left')

    # Drop intermediate columns
    chunk = chunk.drop(columns=['CustomerID', 'SalesPersonID', 'CityID', 'ProductID', 'customer_pk', 'employee_pk', 'city_id', 'product_pk'])
        
    # Rename columns to match fact table
    chunk = chunk.rename(columns={
        'SalesID': 'sales_pk',
        'customer_sk': 'customer_fk',
        'employee_sk': 'employee_fk',
        'country_city_sk': 'city_fk',
        'product_sk': 'product_fk',
        'Quantity': 'quantity',
        'Price': 'price',
        'Discount': 'discount'
    })
        
    return chunk

In [14]:
def load_fact_chunk(chunk, dwh_engine):
    """Load a single chunk into the database"""
    # Use a regular table instead of temp table for better compatibility
    staging_fact = '#temp_fact'
    chunk.to_sql(staging_fact, dwh_engine, if_exists='replace', index=False)
        
    insert_query = """
        insert into Fact_Sales (
            sales_pk, employee_fk, customer_fk, product_fk, city_fk,
            date_key, time_key, quantity, price, discount, total_sales, transaction_number
        )
        select
            src.sales_pk, src.employee_fk, src.customer_fk, src.product_fk, src.city_fk,
            src.date_key, src.time_key, src.quantity, src.price, src.discount,
            src.total_sales, src.transaction_number
        from #temp_fact src
        where not exists (
            SELECT 1 FROM Fact_Sales tgt WHERE tgt.sales_pk = src.sales_pk
        )
    """
        
    with dwh_engine.connect() as conn:
        result = conn.execute(text(insert_query))
        conn.commit()
        return result.rowcount

In [ ]:
def extract_fact_batched(source_engine, dwh_engine):
    print('\n=== Starting Batched Fact Extraction ===\n')
        
    # Get the last loaded sales ID
    last_id = get_last_loaded_sales_id(source_engine)
    print(f"Last loaded Sales ID: {last_id}")
        
    # Get total rows to process
    total_rows = get_total_rows_to_process(last_id, source_engine)
    print(f"Total rows to process: {total_rows:,}")
        
    if total_rows == 0:
        print("No new rows to process.")
        return
        
    # Load dimension lookups once
    dim_lookups = load_dimension_lookups(dwh_engine)
    
    # Set batch size (adjust based on your memory)
    batch_size = 100000
    offset = 0
    total_inserted = 0
    batch_number = 1
        
    while offset < total_rows:
        print(f"\n--- Processing Batch {batch_number} (Rows {offset:,} to {offset + batch_size:,}) ---")
    
        # Fetch one batch
        batch_query = f"""
            select
                s.SalesID,
                s.SalesPersonID,
                s.CustomerID,
                c.CityID,
                s.ProductID,
                s.Quantity,
                s.Discount,
                p.Price,
                cast(format(convert(datetime, s.SalesDate), 'yyyyMMdd') as int) as date_key,
                cast(format(convert(datetime, s.SalesDate), 'HHmmss') as INT) as time_key,
                s.TransactionNumber AS transaction_number
            from GrocerySalesStage.dbo.sales s
            left join GrocerySalesStage.dbo.Customers c ON s.CustomerID = c.CustomerID
            left join GrocerySalesStage.dbo.Products p ON s.ProductID = p.ProductID
            where s.SalesID > {last_id}
            order by s.SalesID
            offset {offset} ROWS
            fetch next {batch_size} rows only
        """
            
        batch_df = pd.read_sql(batch_query, source_engine)
            
        if len(batch_df) == 0:
            print("No more rows to process.")
            break
            
        print(f"Loaded {len(batch_df):,} rows into DataFrame")
            
        # Transform the batch
        transformed_batch = transform_fact_chunk(batch_df, dim_lookups)
            
        # Load the batch
        rows_inserted = load_fact_chunk(transformed_batch, dwh_engine)
        total_inserted += rows_inserted
            
        print(f"Batch {batch_number} complete: {rows_inserted:,} rows inserted")
        print(f"Total inserted so far: {total_inserted:,}")
            
        del batch_df
        del transformed_batch
            
        offset += batch_size
        batch_number += 1
        
    print(f"\n=== Fact Loading Completed: {total_inserted:,} total rows inserted ===")

In [16]:
def Run_ETL_Pipeline():
    
    source, dwh = connections()
    customers_df, employees_df, countries_df, cities_df, products_df, categories_df = extract_dim(source)
    dim_customers, dim_employees, dim_cities, dim_products = transform_dim(customers_df, employees_df, countries_df, cities_df, products_df, categories_df)
    load_dim(dim_customers, dim_employees, dim_cities, dim_products, dwh)
    extract_fact_batched(source, dwh)

    print("\n=======================================")
    print("= ETL Pipeline Completed Successfully =")
    print("=======================================\n")

In [ ]:
if __name__ == "__main__":
    
    Run_ETL_Pipeline()